# Components in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

Alfred is hosting a party and needs to be able to find relevant information on personas that will be attending the party. Therefore, we will use a `QueryEngine` to index and search through a database of personas.

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
!pip install llama-index datasets llama-index-callbacks-arize-phoenix arize-phoenix llama-index-vector-stores-chroma llama-index-llms-google-genai llama-index-embeddings-google-genai -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [ ]:
from huggingface_hub import login

login()

## Create a `QueryEngine` for retrieval augmented generation

### Setting up the persona database

We will be using personas from the [dvilasuero/finepersonas-v0.1-tiny dataset](https://huggingface.co/datasets/dvilasuero/finepersonas-v0.1-tiny). This dataset contains 5K personas that will be attending the party!

Let's load the dataset and store it as files in the `data` directory

In [5]:
from datasets import load_dataset
from pathlib import Path

dataset = load_dataset(path="dvilasuero/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w", encoding="utf-8") as f:
        f.write(persona["persona"])

Awesome, now we have a local directory with all the personas that will be attending the party, we can load and index!

### Loading and embedding persona documents

We will use the `SimpleDirectoryReader` to load the persona descriptions from the `data` directory. This will return a list of `Document` objects. 

In [6]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
len(documents)

5000

Now we have a list of `Document` objects, we can use the `IngestionPipeline` to create nodes from the documents and prepare them for the `QueryEngine`. We will use the `SentenceSplitter` to split the documents into smaller chunks and the `HuggingFaceEmbedding` to embed the chunks.

In [ ]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

import os

gemini_token = os.getenv("GEMINI_KEY")

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        GoogleGenAIEmbedding(model_name="gemini-embedding-2-preview", api_key=gemini_token),
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes

[TextNode(id_='66db10a6-cb95-4c84-9c7f-e4ea2a760a55', embedding=[-0.00526913, 0.01676133, -0.00030602212, 0.002925565, 0.00197502, -0.018643789, -0.022323197, -0.019031994, -0.0007750425, -0.045107313, -0.009651408, 0.0042157415, 0.009711231, -0.011974008, -0.016212609, -0.030205464, 0.013639753, 0.008990813, -0.022010367, 0.0009902017, -0.00097159046, -0.021365741, -0.004716305, 0.02094774, -0.01972174, 0.04080915, -0.00019911307, -0.028280858, -0.010175919, 0.11720121, -0.022898763, -0.010731779, 0.0070326747, 0.016593544, 0.0055894908, -0.0079446975, 0.0037048317, -0.0110015245, 0.020463858, 0.005843788, 0.014997463, 0.024968082, 0.015239003, -0.016660947, -0.00024926531, -0.006490059, -0.02111539, 0.0077218167, 0.013271447, -0.012215628, 0.009424187, -0.007887183, 0.03498321, -0.010741592, 0.010135427, -0.006073768, 0.010308042, -0.005417696, 0.01646518, 0.021888603, -0.012084946, 0.009157439, 0.00659617, -0.014247723, 0.026424835, 0.009425331, 0.013475605, -0.013797889, 0.00048030

As, you can see, we have created a list of `Node` objects, which are just chunks of text from the original documents. Let's explore how we can add these nodes to a vector store.

### Storing and indexing documents

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it.
In this case, we will use `Chroma` to store our documents.
Let's run the pipeline again with the vector store attached. 
The `IngestionPipeline` caches the operations so this should be fast!

In [10]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        GoogleGenAIEmbedding(model_name="gemini-embedding-2-preview", api_key=gemini_token),
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

10

We can create a `VectorStoreIndex` from the vector store and use it to query the documents by passing the vector store and embedding model to the `from_vector_store()` method.

In [11]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding


embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-2-preview", api_key=gemini_token)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

We don't need to worry about persisting the index to disk, as it is automatically saved within the `ChromaVectorStore` object and the passed directory path.

### Querying the index

Now that we have our index, we can use it to query the documents.
Let's create a `QueryEngine` from the index and use it to query the documents using a specific response mode.


In [12]:
from llama_index.llms.google_genai import GoogleGenAI
import nest_asyncio

nest_asyncio.apply()  # This is needed to run the query engine
llm = GoogleGenAI(model_name="models/gemini-3.1-flash-lite", api_key=gemini_token)
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)
response

Response(response='The persona is a local art historian and museum professional who is interested in 19th-century American art and the local cultural heritage of Cincinnati. No information regarding author or travel experiences is available.', source_nodes=[NodeWithScore(node=TextNode(id_='36845b90-e91c-4ae2-82af-44e41af4d7e5', embedding=None, metadata={'file_path': 'd:\\SBT_Study\\SberTechDL\\Agents_Course\\Unit_2\\2_llamaindex\\data\\persona_0.txt', 'file_name': 'persona_0.txt', 'file_type': 'text/plain', 'file_size': 132, 'creation_date': '2026-05-09', 'last_modified_date': '2026-05-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='14e5ec54-1c8e-4ad2-bdba-d5da84c54a5b', node_type='4', metadata={'file

## Evaluation and observability

LlamaIndex provides **built-in evaluation tools to assess response quality.**
These evaluators leverage LLMs to analyze responses across different dimensions.
We can now check if the query is faithful to the original persona.

In [13]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

True

If one of these LLM based evaluators does not give enough context, we can check the response using the Arize Phoenix tool, after creating an account at [LlamaTrace](https://llamatrace.com/login) and generating an API key.

In [14]:
import llama_index
import os

PHOENIX_API_KEY = os.getenv("PHOENIX_API_KEY")
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
llama_index.core.set_global_handler(
    "arize_phoenix", endpoint="https://llamatrace.com/v1/traces"
)


Now, we can query the index and see the response in the Arize Phoenix tool.

In [16]:
response = query_engine.query(
    "What is the name of the someone that is interested in AI and techhnology?"
)
response

Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 
Failed to export span batch code: 401, reason: 


Response(response='There is no information available regarding a person interested in AI and technology.', source_nodes=[NodeWithScore(node=TextNode(id_='36845b90-e91c-4ae2-82af-44e41af4d7e5', embedding=None, metadata={'file_path': 'd:\\SBT_Study\\SberTechDL\\Agents_Course\\Unit_2\\2_llamaindex\\data\\persona_0.txt', 'file_name': 'persona_0.txt', 'file_type': 'text/plain', 'file_size': 132, 'creation_date': '2026-05-09', 'last_modified_date': '2026-05-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='14e5ec54-1c8e-4ad2-bdba-d5da84c54a5b', node_type='4', metadata={'file_path': 'd:\\SBT_Study\\SberTechDL\\Agents_Course\\Unit_2\\2_llamaindex\\data\\persona_0.txt', 'file_name': 'persona_0.txt', 'file_type':

We can then go to the [LlamaTrace](https://llamatrace.com/login) and explore the process and response.

![arize-phoenix](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/arize.png)    